# 분석형 Agent: 코드 리뷰 피드백

PydanticAI를 활용하여 PR diff를 분석하고 보안 취약점, 버그, 성능, 스타일 등을 분류하여 피드백하는 **분석형 Agent**입니다.

**분석형 Agent의 특징:**
- 입력 데이터(PR diff)를 받아 **구조화된 분석 결과**를 반환
- Tool 호출 없이 LLM의 추론 능력만으로 분류/판단 수행
- `output_type`으로 응답 형식을 강제하여 일관된 결과 보장

## 환경 설정

`.env` 파일에서 API 키를 로드하고, PydanticAI Agent 구성에 필요한 라이브러리를 임포트합니다.

In [1]:
# .env 파일에서 OPENAI_API_KEY 등 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

# Pydantic: 구조화된 출력 스키마 정의용
# PydanticAI Agent: LLM 호출 및 도구 오케스트레이션
from pydantic import BaseModel
from pydantic_ai import Agent

## Structured Output

코드 리뷰 결과를 구조화된 형태로 정의합니다.

PydanticAI의 `output_type`에 Pydantic 모델을 지정하면, LLM이 자유 텍스트 대신 **정해진 스키마에 맞는 JSON**을 반환합니다.
이를 통해 후속 처리(Jira 등록, 대시보드 표시 등)가 용이해집니다.

In [2]:
# 개별 피드백 항목: 카테고리, 심각도, 파일, 메시지
class Feedback(BaseModel):
    category: str   # security, bug, performance, style, good_practice
    severity: int   # 1(info) ~ 5(critical)
    file: str       # 피드백 대상 파일 경로
    message: str    # 피드백 내용


# 전체 코드 리뷰 결과: 요약 + 위험도 + 피드백 목록
class CodeReview(BaseModel):
    summary: str              # 리뷰 전체 요약
    risk_level: str           # low, medium, high, critical
    feedbacks: list[Feedback] # 개별 피드백 리스트

## 하드코딩된 PR diff

분석 대상이 되는 샘플 PR diff입니다. 실제 환경에서는 GitHub API Tool로 가져오겠지만,
여기서는 분석형 Agent의 핵심 패턴에 집중하기 위해 하드코딩된 diff를 사용합니다.

이 diff에는 **SQL 인젝션 취약점**, **빈 except 블록**, **해시 비교 개선** 등 다양한 리뷰 포인트가 포함되어 있습니다.

In [3]:
# Agent가 분석할 PR diff (보안 취약점, 좋은 관행 등이 섞인 샘플)
SAMPLE_DIFF = """\
diff --git a/src/auth/login.py b/src/auth/login.py
@@ -12,8 +12,15 @@ class AuthService:
     def login(self, username, password):
-        user = db.query(f"SELECT * FROM users WHERE name='{username}'")
-        if user and user.password == password:
+        user = db.query("SELECT * FROM users WHERE name = %s", (username,))
+        if user and check_password_hash(user.password_hash, password):
             token = jwt.encode({"user_id": user.id}, SECRET_KEY)
             return {"token": token}
-        return {"error": "login failed"}
+        raise AuthenticationError("Invalid credentials")
+    def logout(self, token):
+        try:
+            payload = jwt.decode(token, SECRET_KEY)
+            revoked_tokens.add(token)
+        except:
+            pass

diff --git a/src/api/users.py b/src/api/users.py
+def delete_user(user_id):
+    db.execute(f"DELETE FROM users WHERE id={user_id}")
+    return jsonify({"status": "deleted"})
"""

### [보너스] 실제 GitHub PR diff 가져오기

아래 셀의 주석을 해제하면 hardcoded diff 대신 실제 GitHub PR의 diff를 가져와 분석할 수 있다.
GitHub Token이 있으면 private repo의 PR도 분석 가능하다.

In [10]:
# [보너스] 실제 GitHub PR diff 가져오기
# 아래 주석을 해제하면 실제 GitHub PR의 diff를 가져와 분석합니다.
import httpx, os
owner, repo, pr_number = "fastapi", "fastapi", 15185
headers = {"Accept": "application/vnd.github.v3.diff"}
token = os.getenv("GITHUB_TOKEN")
if token:
    headers["Authorization"] = f"Bearer {token}"
resp = httpx.get(
    f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}",
    headers=headers,
    timeout=10,
)
if resp.status_code == 200:
    SAMPLE_DIFF = resp.text[:3000]  # 앞 3000자만 사용
    print(f"실제 PR diff 로드 완료 ({len(SAMPLE_DIFF)}자)")

실제 PR diff 로드 완료 (3000자)


## Agent 정의

분석형 Agent는 **Tool 없이** LLM의 추론만으로 동작합니다.
`output_type=CodeReview`를 지정하여 응답이 반드시 구조화된 형식으로 반환되도록 강제합니다.

In [8]:
# 분석형 Agent: Tool 없이 output_type만 지정
# LLM이 diff를 읽고 구조화된 CodeReview를 직접 반환
agent = Agent(
    "openai:gpt-5.4",
    output_type=CodeReview,  # 반환 형식을 Pydantic 모델로 강제
    instructions=(
        "시니어 개발자로서 코드 리뷰를 수행한다. "
        "보안 취약점, 버그, 성능, 스타일, 좋은 관행을 분류하여 피드백. "
        "한국어로 작성하되 코드 참조는 영어 유지."
    ),
)

## 실행

PR diff를 Agent에 전달하고 구조화된 코드 리뷰 결과를 받습니다.

> **참고:** Jupyter 노트북은 이미 이벤트 루프가 실행 중이므로 `run_sync()` 대신 `await agent.run()`을 사용합니다.

In [9]:
# PR diff를 Agent에 전달하여 코드 리뷰 실행 (Jupyter에서는 await 사용)
result = await agent.run(
    f"PR #47: 인증 모듈 리팩토링 및 사용자 삭제 API 추가\n\n{SAMPLE_DIFF}"
)
review = result.output  # CodeReview 타입의 구조화된 결과

# 리뷰 요약 및 위험도 출력
print(f"요약: {review.summary}")
print(f"위험도: {review.risk_level.upper()}\n")

# 각 피드백 항목을 카테고리/심각도별로 출력
severity_label = {1: "INFO", 2: "LOW", 3: "MED", 4: "HIGH", 5: "CRIT"}
for fb in review.feedbacks:
    label = severity_label.get(fb.severity, str(fb.severity))
    print(f"  [{fb.category}/{label}] {fb.file}")
    print(f"    {fb.message}\n")

print(f"총 {len(review.feedbacks)}건의 피드백")

요약: 문서 변경은 전반적으로 정확성을 높이는 방향이며, HTTP query parameter에서 `None` 전달 불가라는 핵심 제약을 잘 설명합니다. 다만 표현의 일관성, FastAPI/Pydantic 동작 범위에 대한 명확화, 예시 보강 측면에서 약간의 개선 여지가 있습니다. 코드 변경이 아닌 문서 변경이므로 보안/성능 리스크는 사실상 없습니다.
위험도: LOW

  [좋은 관행/INFO] docs/en/docs/tutorial/query-params-str-validations.md
    추가된 warning 블록은 `Optional[str]` 또는 `str | None` 타입 선언을 실제 HTTP query serialization과 혼동하는 사용자를 줄이는 데 도움이 됩니다. 특히 `?q=null`이 string `"null"`로 처리된다는 설명은 실무적으로 유용합니다.

  [스타일/LOW] docs/en/docs/tutorial/query-params-str-validations.md
    `This means the client must send a value, although the type allows None.` 문장은 더 자연스럽게 `This means the client must send the parameter, even though the annotated type allows None.` 정도로 쓰면 'value'와 'parameter presence'의 차이가 더 분명해집니다.

  [버그/LOW] docs/en/docs/tutorial/query-params-str-validations.md
    `If the parameter is not included at all, FastAPI will raise a validation error because it is required.` 는 문맥상 맞지만, 엄밀히는 FastAPI가 request parsing/validation 단계에서 반환하는 422 validation er